In [2]:
import os, sys, json, math, random
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F

def parse_paths():
    if len(sys.argv) > 2 and not sys.argv[1].startswith('-f'):
        pub, out = Path(sys.argv[1]), Path(sys.argv[2])
    else:
        candidates = ['/kaggle/input/datasets/devansh66/urban-surface-marking',
                      '/kaggle/input/urban-surface-marking',
                      '/kaggle/input', './dataset/public', '.']
        pub = next((Path(c) for c in candidates if (Path(c) / 'train.csv').exists()), Path('./dataset/public'))
        out = Path('./working/submission.csv')
    os.makedirs(out.parent, exist_ok=True)
    return pub, out

PUB, OUT = parse_paths()
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
AMP = DEV == 'cuda'
torch.backends.cudnn.benchmark = True
R, C = 9, 12
IH, IW = 288, 192
FOLDS = 5
EPOCHS = 45

def tok(r, c): return "r%02d_c%02d" % (r, c)

def components(g):
    lab = -np.ones((R, C), int); k = 0
    for i in range(R):
        for j in range(C):
            if g[i, j] and lab[i, j] < 0:
                st = [(i, j)]; lab[i, j] = k
                while st:
                    a, b = st.pop()
                    for da, db in ((1,0),(-1,0),(0,1),(0,-1)):
                        x, y = a+da, b+db
                        if 0 <= x < R and 0 <= y < C and g[x, y] and lab[x, y] < 0:
                            lab[x, y] = k; st.append((x, y))
                k += 1
    return lab, k

class Calib:
    def __init__(self, A):
        self.area = self._lut([(len(a['occupied_cells']), a['area_bin']) for a in A], R*C)
        pr = []
        for a in A:
            if a['marking_count'] == 1:
                k = [k for k, v in a['size_histogram'].items() if v == 1]
                if k: pr.append((len(a['occupied_cells']), ['small','medium','large'].index(k[0])))
        self.sz = self._lut(pr, R*C)
        px, py, pw, ph = [], [], [], []
        for a in A:
            if a['marking_count'] == 1 and len(a['region_cards']) == 1:
                cs = a['occupied_cells']
                r = [int(c[1:3]) for c in cs]; c_ = [int(c[5:7]) for c in cs]
                d = a['region_cards'][0]
                px.append((min(c_)+max(c_), d['cx_bin'])); py.append((min(r)+max(r), d['cy_bin']))
                pw.append((max(c_)-min(c_)+1, d['w_bin'])); ph.append((max(r)-min(r)+1, d['h_bin']))
        self.cx = self._lut(px, 2*C); self.cy = self._lut(py, 2*R)
        self.wb = self._lut(pw, C); self.hb = self._lut(ph, R)

    @staticmethod
    def _lut(pairs, maxk):
        d = defaultdict(Counter)
        for k, v in pairs: d[k][v] += 1
        lut = {k: d[k].most_common(1)[0][0] for k in d}
        ks = sorted(lut)
        return [lut[k] if k in lut else (lut[min(ks, key=lambda x: abs(x-k))] if ks else 0)
                for k in range(maxk+1)]

    def build(self, g):
        g = g.astype(bool)
        occ = [tok(i, j) for i in range(R) for j in range(C) if g[i, j]]
        lab, k = components(g)
        comps = []
        for t in range(k):
            rs, cs = np.where(lab == t)
            comps.append((int(rs.size), rs.min(), rs.max(), cs.min(), cs.max()))
        hist = {'small': 0, 'medium': 0, 'large': 0}; vb = [0]*R
        for s, r0, r1, c0, c1 in comps:
            hist[['small','medium','large'][self.sz[min(s, R*C)]]] += 1
            vb[int(round((r0+r1)/2))] += 1
        hist = {kk: min(v, 9) for kk, v in hist.items()}
        vb = [min(v, 9) for v in vb]
        comps.sort(key=lambda x: -x[0])
        cards = [{'cx_bin': int(self.cx[c0+c1]), 'cy_bin': int(self.cy[r0+r1]),
                  'w_bin': int(self.wb[c1-c0+1]), 'h_bin': int(self.hb[r1-r0+1])}
                 for s, r0, r1, c0, c1 in comps[:12]]
        return {'marking_count': int(min(max(len(comps), 1), 30)), 'occupied_cells': occ,
                'edge_touch_cells': [], 'size_histogram': hist, 'vertical_band_counts': vb,
                'area_bin': int(self.area[min(len(occ), R*C)]), 'region_cards': cards}

def f1(p, t):
    P, T = set(p), set(t)
    if not P and not T: return 1.0
    if not P or not T: return 0.0
    i = len(P & T)
    if i == 0: return 0.0
    return 2*i/(len(P)+len(T))

def dupf1(p, t):
    P = Counter(tuple(sorted(d.items())) for d in p); T = Counter(tuple(sorted(d.items())) for d in t)
    if not P and not T: return 1.0
    if not P or not T: return 0.0
    i = sum((P & T).values())
    if i == 0: return 0.0
    return 2*i/(sum(P.values())+sum(T.values()))

def row_score(p, t):
    return (0.14*max(0, 1-abs(p['marking_count']-t['marking_count'])/8)
            + 0.27*f1(p['occupied_cells'], t['occupied_cells'])
            + 0.12*f1(p['edge_touch_cells'], t['edge_touch_cells'])
            + 0.12*sum(max(0, 1-abs(p['size_histogram'][k]-t['size_histogram'][k])/4) for k in ('small','medium','large'))/3
            + 0.12*sum(max(0, 1-abs(a-b)/5) for a, b in zip(p['vertical_band_counts'], t['vertical_band_counts']))/9
            + 0.08*max(0, 1-abs(p['area_bin']-t['area_bin'])/4)
            + 0.15*dupf1(p['region_cards'], t['region_cards']))

def load_imgs(df):
    X = np.zeros((len(df), IH, IW, 3), np.uint8)
    for i, p in enumerate(df.image_path.values):
        fp = PUB / p
        if not fp.exists(): fp = PUB / 'images' / Path(p).name
        X[i] = np.asarray(Image.open(fp).convert('RGB').resize((IW, IH), Image.BILINEAR))
    return X

train = pd.read_csv(PUB / 'train.csv')
test = pd.read_csv(PUB / 'test.csv')
A = [json.loads(x) for x in train.answer_json]
cal = Calib(A)
Y = np.zeros((len(A), R, C), np.float32)
for i, a in enumerate(A):
    for c in a['occupied_cells']: Y[i, int(c[1:3]), int(c[5:7])] = 1
Xtr = load_imgs(train); Xte = load_imgs(test)

def block(ci, co, s=2):
    return nn.Sequential(nn.Conv2d(ci, co, 3, s, 1, bias=False), nn.BatchNorm2d(co), nn.ReLU(inplace=True),
                         nn.Conv2d(co, co, 3, 1, 1, bias=False), nn.BatchNorm2d(co), nn.ReLU(inplace=True))

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.b = nn.Sequential(block(3, 32), block(32, 64), block(64, 128), block(128, 192))
        self.ctx = nn.Sequential(nn.Conv2d(192, 192, 3, 1, 2, dilation=2, bias=False),
                                 nn.BatchNorm2d(192), nn.ReLU(inplace=True), nn.Dropout2d(0.2))
        self.head = nn.Conv2d(192, 1, 1)
    def forward(self, x):
        x = self.ctx(self.b(x))              # B,192,18,12
        x = F.avg_pool2d(x, (2, 1))          # B,192,9,12
        return self.head(x)[:, 0]

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def to_t(x):
    t = torch.from_numpy(x).float().permute(0, 3, 1, 2) / 255.
    return (t - MEAN) / STD

def augment(xb, yb):
    xb = xb.clone(); yb = yb.clone()
    if random.random() < 0.5:
        xb = torch.flip(xb, [3]); yb = torch.flip(yb, [2])
    if random.random() < 0.8:
        r = lambda: torch.rand(xb.size(0), 1, 1, 1, device=xb.device)
        xb = xb * (0.8 + 0.4*r()) + 0.3*(r()-0.5)
    if random.random() < 0.3:
        xb = xb + 0.05*torch.randn_like(xb)
    return xb, yb

def predict(model, X, bs=64):
    model.eval(); out = []
    with torch.no_grad(), torch.autocast('cuda', enabled=AMP):
        for i in range(0, len(X), bs):
            xb = to_t(X[i:i+bs]).to(DEV, non_blocking=True).contiguous(memory_format=torch.channels_last)
            p = torch.sigmoid(model(xb).float())
            p = p + torch.flip(torch.sigmoid(model(torch.flip(xb, [3])).float()), [2])
            out.append((p/2).cpu().numpy())
    return np.concatenate(out)

idx = np.arange(len(train)); rng = np.random.RandomState(SEED); rng.shuffle(idx)
folds = np.array_split(idx, FOLDS)
oof = np.zeros((len(train), R, C), np.float32)
test_p = np.zeros((len(test), R, C), np.float32)
pw = torch.tensor((1-Y.mean())/Y.mean()*0.6).to(DEV)

for f in range(FOLDS):
    va = folds[f]; tr = np.concatenate([folds[j] for j in range(FOLDS) if j != f])
    model = Net().to(DEV).to(memory_format=torch.channels_last)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-2)
    BS = 48
    steps = EPOCHS * max(1, math.ceil(len(tr)/BS))
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=steps)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    Xf = to_t(Xtr[tr]).to(DEV); Yf = torch.from_numpy(Y[tr]).to(DEV)
    for ep in range(EPOCHS):
        model.train(); perm = torch.randperm(len(tr), device=DEV)
        for i in range(0, len(tr), BS):
            b = perm[i:i+BS]
            xb, yb = augment(Xf[b], Yf[b])
            xb = xb.contiguous(memory_format=torch.channels_last)
            with torch.autocast('cuda', enabled=AMP):
                loss = F.binary_cross_entropy_with_logits(model(xb), yb, pos_weight=pw)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step()
    oof[va] = predict(model, Xtr[va])
    test_p += predict(model, Xte)/FOLDS
    print('fold', f, 'loss', loss.item(), flush=True)

best, bth = -1, 0.5
for th in np.arange(0.20, 0.76, 0.02):
    s = np.mean([row_score(cal.build(oof[i] >= th), A[i]) for i in range(len(A))])
    if s > best: best, bth = s, th
print('OOF score %.4f @ th=%.2f' % (best, bth), flush=True)

rows = []
for i, rid in enumerate(test.id.values):
    g = test_p[i] >= bth
    if g.sum() == 0:
        g = test_p[i] >= np.sort(test_p[i].ravel())[-4]
    rows.append({'id': rid, 'answer_json': json.dumps(cal.build(g), separators=(',', ':'))})
pd.DataFrame(rows).to_csv(OUT, index=False)
print('wrote', OUT, len(rows))

fold 0 loss 0.01607179082930088
fold 1 loss 0.0250324085354805
fold 2 loss 0.02503778412938118
fold 3 loss 0.02350638620555401
fold 4 loss 0.03472869470715523
OOF score 0.6290 @ th=0.38
wrote working/submission.csv 322
